# Notebook 2: RFM Analysis & Customer Segmentation (K-Means)

This notebook implements RFM (Recency, Frequency, Monetary) analysis and K-Means clustering to segment customers into actionable groups. RFM is a behavioral segmentation technique that helps identify high-value customers, at-risk customers, and dormant accounts.

**Author:** Stephen Drani  
**Date:** March 2026

## 2.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings

warnings.filterwarnings('ignore')

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## 2.2 Load Cleaned Data

In [ ]:
# Load the cleaned marketing data from Notebook 01
df = pd.read_csv('../data/processed/marketing_cleaned.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names and types:")
print(df.dtypes)
print(f"\nFirst few rows:")
df.head()

## 2.3 RFM Score Calculation

RFM is a customer segmentation technique based on three metrics:

- **Recency (R):** How recently a customer made a purchase (days ago). Lower values are better (more recent = higher score). We invert this by using labels [5,4,3,2,1].
- **Frequency (F):** How often a customer has made a purchase (Total_Purchases). Higher values are better (more purchases = higher score).
- **Monetary (M):** How much a customer has spent in total (MntTotal). Higher values are better (more spending = higher score).

Each metric is scored from 1 to 5 using quintiles, where 5 is the best/most valuable segment.
We combine these into an RFM_Score string (e.g., "555" = Champions, "111" = Hibernating).

In [ ]:
# Calculate RFM Scores using quintiles

# Recency: Lower is better (more recent), so we invert the score
# Using qcut with labels=False + 1 to get scores 1-5, then invert
r_temp = pd.qcut(df['Recency'], q=5, labels=False, duplicates='drop') + 1
df['R_Score'] = 6 - r_temp  # Invert so lower recency gets higher score
df['R_Score'] = df['R_Score'].astype(int)

# Frequency: Higher is better (use Total_Purchases)
f_temp = pd.qcut(df['Total_Purchases'], q=5, labels=False, duplicates='drop') + 1
df['F_Score'] = f_temp
df['F_Score'] = df['F_Score'].astype(int)

# Monetary: Higher is better (use MntTotal)
m_temp = pd.qcut(df['MntTotal'], q=5, labels=False, duplicates='drop') + 1
df['M_Score'] = m_temp
df['M_Score'] = df['M_Score'].astype(int)

# Create RFM_Score string (e.g., "555")
df['RFM_Score'] = df['R_Score'].astype(str) + df['F_Score'].astype(str) + df['M_Score'].astype(str)

print("RFM Score Distribution:")
print(f"\nRecency Score (5=Most Recent):")
print(df['R_Score'].value_counts().sort_index(ascending=False))
print(f"\nFrequency Score (5=Most Frequent):")
print(df['F_Score'].value_counts().sort_index(ascending=False))
print(f"\nMonetary Score (5=Highest Spend):")
print(df['M_Score'].value_counts().sort_index(ascending=False))
print(f"\nSample RFM Scores:")
print(df[['Recency', 'Total_Purchases', 'MntTotal', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score']].head(10))

## 2.4 RFM Segment Mapping

Map RFM scores to 8 customer segments based on behavioral patterns:

In [ ]:
def rfm_segment(score):
    """
    Map RFM score to customer segment (8 segments)
    """
    if score == '555':
        return 'Champions'
    elif score in ['554', '544', '545', '454', '455', '445', '555']:
        return 'Loyal'
    elif score in ['543', '444', '443', '442', '441', '451', '452']:
        return 'Potential Loyalists'
    elif score in ['512', '511', '522', '521', '531', '532']:
        return 'New Customers'
    elif score in ['525', '535', '545', '435', '325', '424', '425']:
        return 'Promising'
    elif score in ['313', '314', '323', '324', '332', '342', '343']:
        return 'Need Attention'
    elif score in ['153', '163', '255', '265', '333', '322', '331', '341']:
        return 'At Risk'
    else:
        return 'Hibernating'

# Apply segment mapping
df['RFM_Segment'] = df['RFM_Score'].apply(rfm_segment)

print("RFM Segment Distribution:")
segment_counts = df['RFM_Segment'].value_counts()
print(segment_counts)
print(f"\nSegment Percentages:")
print((segment_counts / len(df) * 100).round(2))

# Visualize segment sizes
plt.figure(figsize=(12, 6))
ax = segment_counts.plot(kind='barh', color='steelblue')
plt.title('Customer Distribution by RFM Segment', fontsize=14, fontweight='bold')
plt.xlabel('Number of Customers', fontsize=12)
plt.ylabel('RFM Segment', fontsize=12)
plt.tight_layout()
plt.savefig('../visualizations/01_rfm_segment_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nSegment distribution chart saved.")

## 2.5 RFM Segment Profiles

In [ ]:
# Create RFM segment profiles
rfm_profiles = df.groupby('RFM_Segment').agg({
    'Recency': 'mean',
    'Total_Purchases': 'mean',
    'MntTotal': 'mean',
    'Income': 'mean'
}).round(2)

# Sort by MntTotal descending
rfm_profiles = rfm_profiles.sort_values('MntTotal', ascending=False)

# Rename columns for better presentation
rfm_profiles.columns = ['Avg Recency (days)', 'Avg Frequency', 'Avg Monetary ($)', 'Avg Income ($)']

print("RFM Segment Profiles:")
print(rfm_profiles)

# Create styled summary table
styled_rfm = rfm_profiles.style.background_gradient(cmap='RdYlGn', axis=0)
print("\nStyled RFM Profiles (color-coded):")
styled_rfm

## 2.6 Preparing for K-Means Clustering

K-Means clustering requires numerical features to be on the same scale. We use StandardScaler to normalize the RFM scores to have mean=0 and std=1. This ensures that features with larger ranges don't dominate the clustering algorithm.

In [ ]:
# Select RFM scores for clustering (not raw features)
clustering_features = ['R_Score', 'F_Score', 'M_Score']

# Create a copy for scaling
X = df[clustering_features].copy()

print("Before Scaling:")
print(X.describe().round(2))

# Fit and transform with StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=clustering_features)

print("\nAfter Scaling (StandardScaler):")
print(X_scaled_df.describe().round(2))

print("\nRFM scores are now standardized with mean~0 and std~1")

## 2.7 Finding Optimal K (Elbow Method + Silhouette)

In [ ]:
# Test K from 2 to 8
k_range = range(2, 9)
inertias = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

print("K-Means Evaluation Metrics:")
for k, inertia, sil_score in zip(k_range, inertias, silhouette_scores):
    print(f"K={k}: Inertia={inertia:.2f}, Silhouette Score={sil_score:.4f}")

# Plot Elbow Method and Silhouette Score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Method
axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inertia (Within-cluster sum of squares)', fontsize=12)
axes[0].set_title('Elbow Method for Optimal K', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(k_range)

# Silhouette Score
axes[1].plot(k_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score for Optimal K', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(k_range)

plt.tight_layout()
plt.savefig('../visualizations/02_optimal_k_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nOptimal K analysis chart saved.")

## 2.8 Fit K-Means with Optimal K

In [ ]:
# Use K=4 based on elbow and silhouette analysis
optimal_k = 4

# Fit KMeans with optimal K
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['Cluster'] = kmeans_final.fit_predict(X_scaled)

print(f"K-Means clustering completed with K={optimal_k}")
print(f"\nCluster Distribution:")
print(df['Cluster'].value_counts().sort_index())
print(f"\nCluster Percentages:")
print((df['Cluster'].value_counts(normalize=True).sort_index() * 100).round(2))

## 2.9 Cluster Profiles

In [ ]:
# Create detailed cluster profiles
cluster_profiles = df.groupby('Cluster').agg({
    'Recency': 'mean',
    'Total_Purchases': 'mean',
    'MntTotal': 'mean',
    'Income': 'mean',
    'Age': 'mean',
    'Children': 'mean'
}).round(2)

# Sort by MntTotal descending
cluster_profiles = cluster_profiles.sort_values('MntTotal', ascending=False)

# Rename columns for better presentation
cluster_profiles.columns = [
    'Avg Recency (days)',
    'Avg Frequency',
    'Avg Monetary ($)',
    'Avg Income ($)',
    'Avg Age',
    'Avg Children'
]

print("K-Means Cluster Profiles:")
print(cluster_profiles)

# Create styled summary table
styled_clusters = cluster_profiles.style.background_gradient(cmap='viridis', axis=0)
print("\nStyled Cluster Profiles (color-coded):")
styled_clusters

In [ ]:
# Create a grouped bar chart comparing clusters
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

metrics = ['Avg Recency (days)', 'Avg Frequency', 'Avg Monetary ($)', 'Avg Income ($)', 'Avg Age', 'Avg Children']
colors = plt.cm.Set2(np.linspace(0, 1, optimal_k))

for idx, metric in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    cluster_profiles[metric].plot(kind='bar', ax=ax, color=colors)
    ax.set_title(f'{metric} by Cluster', fontsize=12, fontweight='bold')
    ax.set_xlabel('Cluster', fontsize=10)
    ax.set_ylabel(metric.split('(')[0].strip(), fontsize=10)
    ax.tick_params(axis='x', rotation=0)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../visualizations/03_cluster_profiles_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Cluster profiles comparison chart saved.")

## 2.10 Cluster Visualization (PCA)

In [ ]:
# 2D Visualization using PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(12, 8))
colors = plt.cm.Set2(np.linspace(0, 1, optimal_k))

for cluster in range(optimal_k):
    mask = df['Cluster'] == cluster
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], 
               c=[colors[cluster]], 
               label=f"Cluster {cluster}",
               s=100, alpha=0.7, edgecolors='black', linewidth=0.5)

# Plot cluster centers
centers_pca = pca.transform(kmeans_final.cluster_centers_)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], 
           c='red', marker='X', s=300, edgecolors='black', linewidth=2, label='Cluster Centers')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.title(f'K-Means Clustering Results (K={optimal_k}, PCA Visualization)', fontsize=14, fontweight='bold')
plt.legend(fontsize=10, loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/04_cluster_visualization_pca.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"PCA explains {pca.explained_variance_ratio_.sum():.1%} of total variance.")
print("Cluster visualization saved.")

## 2.11 Cluster vs RFM Segment Comparison

In [ ]:
# Cross-tabulation of K-Means clusters vs RFM segments
crosstab = pd.crosstab(df['Cluster'], df['RFM_Segment'])

print("Cross-tabulation of K-Means Clusters vs RFM Segments:")
print(crosstab)

# Heatmap showing overlap
plt.figure(figsize=(14, 6))
sns.heatmap(crosstab, annot=True, fmt='d', cmap='YlOrRd', cbar_kws={'label': 'Count'})
plt.title('Overlap Between K-Means Clusters and RFM Segments', fontsize=14, fontweight='bold')
plt.xlabel('RFM Segment', fontsize=12)
plt.ylabel('K-Means Cluster', fontsize=12)
plt.tight_layout()
plt.savefig('../visualizations/05_cluster_rfm_comparison_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nCluster vs RFM comparison heatmap saved.")

## 2.12 Save Segmented Data

In [ ]:
# Save dataframe with cluster labels
output_path = '../data/processed/marketing_segmented.csv'
df.to_csv(output_path, index=False)

print(f"Segmented data saved to: {output_path}")
print(f"\nDataframe shape: {df.shape}")
print(f"\nColumns included:")
print(df.columns.tolist())
print(f"\nNew segmentation columns:")
print(f"  - R_Score, F_Score, M_Score: Individual RFM scores (1-5)")
print(f"  - RFM_Score: Combined R, F, M scores (e.g., '555')")
print(f"  - RFM_Segment: Named RFM segment (8 categories)")
print(f"  - Cluster: K-Means cluster assignment (0-{optimal_k-1})")

## 2.13 Key Findings

**Summary of Customer Segmentation Analysis**

Please summarize your key findings here after reviewing all the visualizations and tables above:

- [ ] Which RFM segment or K-Means cluster represents the highest value customers?
- [ ] Which segments show at-risk behavior (high recency, low frequency)?
- [ ] What are the main differences between the K-Means clusters?
- [ ] Are there any surprising patterns in the data?
- [ ] What are the recommended actions for each segment?

*Add your findings below:*

---